# Fine-tuning QLoRA do Qwen2.5-3B-Instruct (Google Colab)

Versão "grande" do projeto: o mesmo conjunto de dados da biblioteca fictícia, agora com o **Qwen2.5-3B** carregado em **4 bits (QLoRA)** numa GPU do Colab.

Antes de rodar: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**.
O Colab gratuito não garante GPU em toda sessão.

> Este notebook não foi executado pelo autor (o treino de verdade do repositório foi feito na CPU com o modelo 0.5B). As bibliotecas mudam rápido: se der erro de versão no `SFTConfig`, ajuste os nomes dos parâmetros para a versão instalada.

In [ ]:
!nvidia-smi
import torch
print("CUDA disponível:", torch.cuda.is_available())

In [ ]:
!pip install -q unsloth datasets

In [ ]:
import json, urllib.request

def baixar(nome):
    return urllib.request.urlopen("https://raw.githubusercontent.com/Phb29/finetuning-qwen-biblioteca/main/dados/" + nome).read().decode("utf-8")

TELAS = json.loads(baixar("telas.json"))
treino = [json.loads(l) for l in baixar("treino.jsonl").splitlines() if l.strip()]
teste = [json.loads(l) for l in baixar("teste.jsonl").splitlines() if l.strip()]
print(len(TELAS), "telas |", len(treino), "exemplos de treino |", len(teste), "de teste")

In [ ]:
PROMPT_SISTEMA = (
    "Você é o assistente de um sistema de biblioteca. "
    "Leia o pedido do funcionário e indique somente a tela mais adequada, "
    "no formato: Abra a tela <nome da tela>.\n"
    "Telas disponíveis: " + "; ".join(TELAS) + "."
)

def mensagens(pergunta, tela=None):
    m = [{"role": "system", "content": PROMPT_SISTEMA}, {"role": "user", "content": pergunta}]
    if tela:
        m.append({"role": "assistant", "content": f"Abra a tela {tela}."})
    return m

import unicodedata
def normalizar(t):
    return " ".join(unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode().lower().split())

def tela_citada(resposta):
    r = normalizar(resposta)
    achadas = [(r.find(normalizar(t)), t) for t in TELAS if normalizar(t) in r]
    return min(achadas)[1] if achadas else None

## Carregar o Qwen2.5-3B em 4 bits (o "Q" do QLoRA)

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
    full_finetuning = False,
)

In [ ]:
import torch

def avaliar(model, rotulo):
    FastModel.for_inference(model)
    acertos = 0
    for item in teste:
        entrada = tokenizer.apply_chat_template(
            mensagens(item["pergunta"]), add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
        saida = model.generate(input_ids=entrada, max_new_tokens=32, do_sample=False)
        resposta = tokenizer.decode(saida[0][entrada.shape[-1]:], skip_special_tokens=True).strip()
        ok = tela_citada(resposta) == item["tela"]
        acertos += ok
        print("OK  " if ok else "ERRO", "|", item["pergunta"], "->", resposta)
    print(f"\n{rotulo}: {acertos}/{len(teste)} ({100*acertos/len(teste):.1f}%)")

avaliar(model, "Original")

## Adicionar os adaptadores LoRA e treinar

In [ ]:
model = FastModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

from datasets import Dataset
dataset = Dataset.from_list([
    {"text": tokenizer.apply_chat_template(mensagens(i["pergunta"], i["tela"]), tokenize=False)}
    for i in treino
])
print(dataset[0]["text"])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    processing_class = tokenizer,
    args = SFTConfig(
        output_dir = "qwen3b-qlora-output",
        dataset_text_field = "text",
        max_seq_length = 1024,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)
trainer.train()

## Avaliar de novo nas MESMAS perguntas inéditas e salvar

In [ ]:
avaliar(model, "Com QLoRA")

In [ ]:
model.save_pretrained("qwen25-3b-biblioteca-qlora")
tokenizer.save_pretrained("qwen25-3b-biblioteca-qlora")
!zip -rq qwen25-3b-biblioteca-qlora.zip qwen25-3b-biblioteca-qlora
print("Baixe o .zip pelo painel Arquivos do Colab.")